# Mining the DuckChain 🦆⛏️

In this notebook, we'll explore how blockchains create consensus, perform
computational work, and efficiently verify transactions.

We'll use code to explore:

1. **Proof of Work** — mining blocks by searching for a matching hash
2. **Mining Cost** — predicting the time and computational work required
3. **Proof of Stake** — selecting validators based on economic stake
4. **Merkle Trees** — summarizing many transactions with a single root hash
5. **Merkle Proofs** — efficiently proving that a transaction is included

The examples move from mining and consensus to the data structures that help
blockchains efficiently organize and verify transactions.

> **Goal:** Understand the work, consensus, and verification mechanisms behind
> a blockchain.

## Example 1 — Mining a Block with Proof of Work

So far, DuckChain has calculated a block's hash once. In a **proof-of-work**
blockchain, miners repeatedly calculate hashes while searching for one that
satisfies a difficulty requirement.

This example adds two ideas:

- A **nonce** is a number the miner changes to produce a different hash.
- **Difficulty** specifies how many zeros the hash must begin with in our
  simplified mining rule.

The `mine_block()` method repeatedly increments the nonce and recalculates the
block's hash until the target is reached.

Try different difficulty values and observe how the number of attempts and
mining time change.

> **Goal:** See how changing a nonce turns hashing into computational work.

In [ ]:
# Example 1 - mine a block with Proof of Work

import hashlib
import json
import time

def sha256(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

class Block:
    def __init__(self, index, data):
        self.index = index
        self.data = data
        self.previous_hash = None
        self.nonce = 0
        self.hash = self.calculate_hash()

    # The nonce becomes part of the block's digital fingerprint
    def calculate_hash(self):
        block_text = (
            str(self.index) +
            json.dumps(self.data, sort_keys=True) +
            str(self.previous_hash) +
            str(self.nonce)
        )
        return sha256(block_text)

    # Keep changing the nonce until the hash meets the target
    def mine_block(self, difficulty):
        target = "0" * difficulty
        attempts = 0
        start_time = time.time()

        while True:
            self.hash = self.calculate_hash()
            attempts += 1  # One attempt = one hash calculation

            if self.hash.startswith(target):
                break

            self.nonce += 1  # New nonce produces a new hash

        elapsed_time = time.time() - start_time

        print(f"Block mined: {self.hash}")
        print(f"Nonce: {self.nonce:,}")
        print(f"Attempts: {attempts:,}")
        print(f"Time: {elapsed_time:.3f} seconds")


class Blockchain:
    def __init__(self, difficulty=3):
        self.chain = [self.create_genesis_block()]
        self.difficulty = difficulty

    def create_genesis_block(self):
        return Block(0, "Genesis Block Quack Quack!")

    def get_latest_block(self):
        return self.chain[-1]

    # Link → mine → add
    def add_block(self, new_block):
        new_block.previous_hash = self.get_latest_block().hash
        new_block.hash = new_block.calculate_hash()
        new_block.mine_block(self.difficulty)
        self.chain.append(new_block)


# Create DuckChain with a target of 3 leading zeros
duckchain = Blockchain(difficulty=3)

# Create a transaction block and mine it
duckchain.add_block(Block(
    1, {"transaction": "Donald Duck sent 5 quackers to Daffy Duck"}
))

# Examine the successfully mined block
block = duckchain.get_latest_block()

print("\nMined Block")
print("Index:", block.index)
print("Transaction:", block.data["transaction"])
print("Previous hash:", block.previous_hash)
print("Nonce:", block.nonce)
print("Hash:", block.hash)

## Example 2 — Predicting the Cost of Proof of Work

Mining is a search problem. Each hexadecimal character in a SHA-256 hash has
16 possible values, so requiring another leading zero makes a matching hash
about **16× harder to find**.

For a difficulty of `d`, the average number of hashes needed is:

**16ᵈ**

In this experiment, we'll:

- Measure how many hashes this computer can calculate per second
- Choose a mining difficulty
- Predict the average number of hashes required
- Estimate how long mining should take
- Mine a block using Example 1 and compare the result

Mining is based on chance, so a block may be found much sooner or later than
predicted. The prediction represents the **average over many mining attempts**.

> **Goal:** Connect mining difficulty to computational work and time.

In [ ]:
# Example 2 - predict the cost of Proof of Work

# Create a test block
test_block = Block(
    2, {"transaction": "Daffy Duck sent 3 quackers to Daisy Duck"}
)

# Measure actual block-hashing speed
TEST_HASHES = 100_000
start = time.time()

for i in range(TEST_HASHES):
    test_block.nonce = i
    test_block.calculate_hash()

elapsed = time.time() - start
hash_rate = TEST_HASHES / elapsed

# Choose a mining difficulty
DIFFICULTY = 4

# Each leading hex zero makes a match 16x less likely
expected_hashes = 16 ** DIFFICULTY
expected_time = expected_hashes / hash_rate

print("Predicted Mining Cost")
print(f"Difficulty:      {DIFFICULTY}")
print(f"Hash rate:       {hash_rate:,.0f} hashes/sec")
print(f"Expected hashes: {expected_hashes:,}")
print(f"Expected time:   {expected_time:.3f} seconds")

# Reset the nonce before mining
test_block.nonce = 0
test_block.hash = test_block.calculate_hash()

print("\nActual Mining Result")
test_block.mine_block(DIFFICULTY)

## Example 3 — Building a Proof-of-Stake DuckChain

Proof of Work uses computational work to determine who can add the next
block. **Proof of Stake (PoS)** takes a different approach: validators put
something valuable at stake, and that stake can influence who is selected
to propose the next block.

In this simplified DuckChain:

- Huey stakes 10 quackers
- Dewey stakes 30 quackers
- Louie stakes 60 quackers
- A validator is randomly selected, weighted by its stake
- The selected validator adds the next block without mining it

We'll first run the selection 1,000 times to see how stake affects the
probability of being selected. Then we'll select a validator to add an
actual block to DuckChain.

Real Proof-of-Stake systems include additional rules for validation,
rewards, penalties, and consensus. This example focuses on the basic idea.

> **Goal:** Compare computational work in Proof of Work with economic stake
> in Proof of Stake.

In [ ]:
# Example 3 - build a simple Proof-of-Stake DuckChain

import hashlib
import json
import random

def sha256(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

class Block:
    def __init__(self, index, data, previous_hash=None):
        self.index = index
        self.data = data
        self.previous_hash = previous_hash
        self.validator = None
        self.hash = self.calculate_hash()

    def calculate_hash(self):
        text = (str(self.index) +
                json.dumps(self.data, sort_keys=True) +
                str(self.previous_hash) +
                str(self.validator))
        return sha256(text)

class Validator:
    def __init__(self, name, stake):
        self.name = name
        self.stake = stake

# Select randomly, with probability weighted by stake
def select_validator(validators):
    return random.choices(
        validators,
        weights=[v.stake for v in validators],
        k=1
    )[0]

class StakeBlockchain:
    def __init__(self):
        self.chain = [Block(0, "Genesis Block Quack Quack!")]

    # Link and add the block - no mining required
    def add_block(self, block, validator):
        block.previous_hash = self.chain[-1].hash
        block.validator = validator.name
        block.hash = block.calculate_hash()
        self.chain.append(block)


# Create three validators with different stakes
validators = [
    Validator("Huey", 10),
    Validator("Dewey", 30),
    Validator("Louie", 60)
]

# See how stake affects selection over many trials
TRIALS = 1000
wins = {v.name: 0 for v in validators}

for _ in range(TRIALS):
    winner = select_validator(validators)
    wins[winner.name] += 1

print("Validator Selection")
for v in validators:
    percent = 100 * wins[v.name] / TRIALS
    print(f"{v.name:6} Stake: {v.stake:2}  "
          f"Selected: {percent:5.1f}%")

# Now select a validator to add the next block
winner = select_validator(validators)

duckchain = StakeBlockchain()
new_block = Block(
    1, {"transaction": "Donald sent 5 quackers to Daffy"}
)

duckchain.add_block(new_block, winner)

print("\nNext Block")
print("Selected validator:", winner.name)
print("Transaction:", new_block.data["transaction"])
print("Previous hash:", new_block.previous_hash)
print("Block hash:", new_block.hash)
print("Blocks in DuckChain:", len(duckchain.chain))

## Example 4 — Merkle Trees and Proofs

A **Merkle tree** uses hashes to summarize many transactions with a single
**Merkle root**.

In this example, we'll:

- Hash four transactions to create the leaves of a Merkle tree
- Combine pairs of hashes until one Merkle root remains
- Create a small **Merkle proof** for one transaction
- Verify the transaction by reconstructing the Merkle root

Merkle proofs allow a transaction to be verified without examining every
transaction in the block.

**The Tree — Proving Tx2**

<pre>
             Merkle Root
             /         \
         <b>H01</b>         H23
          /   \       /   \
        H0     H1   <b>H2</b>    <b>H3</b>
        |      |     |      |
       Tx0    Tx1   <b><i>Tx2</i></b>    Tx3
</pre>

***Tx2*** is the transaction being proved. **H2** is calculated from Tx2,
while **H3** and **H01** are the sibling hashes supplied by the Merkle proof.

> **Goal:** See how hashes form a Merkle tree and provide efficient proof that
> a transaction is included.

In [ ]:
# Example 4 - build a simple Merkle tree and proof

import hashlib

def sha256(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

# Four transactions become the leaves of our tree
transactions = [
    "Donald pays Daisy 5",
    "Daisy pays Daffy 3",
    "Daffy pays Huey 2",
    "Huey pays Louie 1"
]

# Hash each transaction
h0 = sha256(transactions[0])
h1 = sha256(transactions[1])
h2 = sha256(transactions[2])
h3 = sha256(transactions[3])

# Combine neighboring hashes
h01 = sha256(h0 + h1)
h23 = sha256(h2 + h3)

# One hash summarizes the entire tree
merkle_root = sha256(h01 + h23)

print("Merkle root:", merkle_root)

# Prove Tx2 is included using only its sibling and the other branch
proof_h23 = sha256(h2 + h3)
proof_root = sha256(h01 + proof_h23)

print("Tx2 proof valid:", proof_root == merkle_root)

## Example 5 — Build, Verify, and Visualize a Merkle Proof

A **Merkle proof** uses a small set of sibling hashes to prove that a
transaction belongs to a Merkle tree without examining every transaction.

In this example, we'll:

- Build a Merkle tree from eight transactions
- Pick any transaction to prove
- Collect the **sibling hash** needed at each level
- Reconstruct the **Merkle root** to verify the proof
- Visualize the selected transaction, proof siblings, and path to the root

Change `TRANSACTION_TO_PROVE` to explore different transactions and see how
the proof path changes.

With eight transactions, only **3 sibling hashes** are needed:
`log₂(8) = 3`.

The visualization uses **green** for the selected transaction and calculated
path, **gold** for sibling hashes supplied by the proof, and **white** for the
rest of the Merkle tree.

> **Goal:** See how a few sibling hashes can prove that a transaction belongs
> to a much larger set.

In [ ]:
# Build, verify, and visualize a Merkle proof

import hashlib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

def sha256(text):
    return hashlib.sha256(text.encode()).hexdigest()

# Build all levels: leaves → branches → root
def build_merkle_tree(transactions):
    tree = [[sha256(tx) for tx in transactions]]
    while len(tree[-1]) > 1:
        level = tree[-1]
        tree.append([
            sha256(level[i] + level[i + 1])
            for i in range(0, len(level), 2)
        ])
    return tree

# Collect one sibling hash at each level
def create_proof(tree, tx_index):
    proof, index = [], tx_index
    for level in tree[:-1]:
        sibling = index + 1 if index % 2 == 0 else index - 1
        proof.append((level[sibling], sibling < index))
        index //= 2
    return proof

# Reconstruct the root from a transaction and its proof
def verify_proof(transaction, proof, merkle_root):
    current = sha256(transaction)
    for sibling, sibling_first in proof:
        current = sha256(
            sibling + current if sibling_first else current + sibling
        )
    return current == merkle_root

# Draw the tree, calculated path, and supplied sibling hashes
def visualize_proof(tree, transactions, tx_index):
    fig, ax = plt.subplots(figsize=(14, 7))
    levels, positions = len(tree), {}

    for level, nodes in enumerate(tree):
        gap = 2 ** level
        for i in range(len(nodes)):
            positions[level, i] = (i * gap + (gap - 1) / 2, level + 1)

    path, siblings, index = set(), set(), tx_index
    for level in range(levels):
        path.add((level, index))
        if level < levels - 1:
            siblings.add((level, index + 1 if index % 2 == 0 else index - 1))
        index //= 2

    for level in range(levels - 1):
        for i in range(len(tree[level])):
            x1, y1 = positions[level, i]
            x2, y2 = positions[level + 1, i // 2]
            ax.plot([x1, x2], [y1, y2], color="lightgray")

    for level, nodes in enumerate(tree):
        for i in range(len(nodes)):
            x, y = positions[level, i]
            sibling, calculated = (level, i) in siblings, (level, i) in path
            color = "gold" if sibling else "lightgreen" if calculated else "white"
            edge = "black" if sibling or calculated else "gray"

            if level == 0:
                label = f"H{i}"
            elif level == levels - 1:
                label = "MR"
            else:
                width = 2 ** level
                start = i * width
                label = f"H{start}{start + width - 1}"

            ax.text(x, y, label, ha="center", va="center",
                    bbox=dict(boxstyle="round,pad=.35",
                              fc=color, ec=edge))

    for i in range(len(transactions)):
        x, _ = positions[0, i]
        selected = i == tx_index
        ax.text(x, 0, f"Tx{i}", ha="center", va="center",
                bbox=dict(boxstyle="round,pad=.3",
                          fc="lightgreen" if selected else "white",
                          ec="black" if selected else "gray"))

    key = [
        Patch(fc="lightgreen", ec="black",
              label="Selected transaction + calculated path"),
        Patch(fc="gold", ec="black",
              label="Sibling hashes supplied by proof"),
        Patch(fc="white", ec="gray", label="Rest of Merkle tree")
    ]
    ax.legend(handles=key, loc="upper left", frameon=False)
    ax.set_title(f"Merkle Proof for Tx{tx_index}: {transactions[tx_index]}")
    ax.set_xlim(-.7, len(transactions) - .3)
    ax.set_ylim(-.6, levels + 1.2)
    ax.axis("off")
    plt.show()

# Run the example

transactions = [
    "Donald pays Daisy 5",
    "Daisy pays Daffy 3",
    "Daffy pays Huey 2",
    "Huey pays Louie 1",
    "Louie pays Dewey 4",
    "Dewey pays Donald 2",
    "Scrooge pays Donald 10",
    "Donald pays Huey 1"
]

# Pick any transaction to prove
TRANSACTION_TO_PROVE = 2

tree = build_merkle_tree(transactions)
merkle_root = tree[-1][0]

transaction = transactions[TRANSACTION_TO_PROVE]
proof = create_proof(tree, TRANSACTION_TO_PROVE)

print("Transaction:", transaction)
print("Merkle root:", merkle_root)

print("\nSibling hashes supplied for proof:")
for level, (sibling_hash, _) in enumerate(proof):
    print(f"Level {level}: {sibling_hash[:16]}...")

print("\nProof valid:",
      verify_proof(transaction, proof, merkle_root))

visualize_proof(tree, transactions, TRANSACTION_TO_PROVE)